## Phase 0.2 – Effect of the train/test split on the late delivery model

The same Random Forest classifier is trained and evaluated with three different splits:

1. **Random rows (original approach):** 80/20 stratified random split of order items.
2. **Grouped by Order Id:** all items of an order are placed either in the training or in the test set.
3. **Time-based:** the model is trained on the oldest 80% of orders and tested on the most recent 20%.

Items belonging to the same order almost always share the same label. With a random row split, items of one order end up in both sets, so the model can effectively "recognise" the order instead of learning a generalisable pattern (data leakage). The time-based split reflects real use: the model learns from the past and predicts the future. Therefore, the time-based result is reported as the realistic performance estimate.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit

from src.data import load_data
from src.features import DELAY_FEATURES, DELAY_TARGET, delay_preprocessor
from src.evaluation import time_split, classification_metrics

In [2]:
data = load_data()
print(data.shape)

(180519, 51)


In [3]:
X = data[DELAY_FEATURES]
y = data[DELAY_TARGET]
groups = data["Order Id"]
print("Raw feature columns:", X.shape[1])

Raw feature columns: 12


In [4]:
def make_model():
    return Pipeline([
        ("prep", delay_preprocessor()),
        ("model", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ])

def evaluate(train_idx, test_idx, name):
    model = make_model().fit(X.iloc[train_idx], y.iloc[train_idx])
    proba = model.predict_proba(X.iloc[test_idx])[:, 1]
    return {"split": name, **classification_metrics(y.iloc[test_idx], proba)}

all_idx = np.arange(len(data))
results = []

tr, te = train_test_split(all_idx, test_size=0.2, random_state=42, stratify=y)
results.append(evaluate(tr, te, "1) random rows (old)"))

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(X, y, groups))
results.append(evaluate(tr, te, "2) grouped by Order Id"))

tr, te = time_split(data)
results.append(evaluate(tr, te, "3) time-based (last 20%)"))

print(pd.DataFrame(results).round(3).to_string(index=False))

                   split  accuracy  precision_late  recall_late  f1_late  roc_auc  pr_auc
    1) random rows (old)     0.808           0.849        0.790    0.818    0.899   0.924
  2) grouped by Order Id     0.681           0.730        0.663    0.695    0.750   0.818
3) time-based (last 20%)     0.696           0.756        0.656    0.702    0.762   0.825


In [5]:
# Train on the time-based training set, then score ONE raw order,
# exactly the way the API will receive it.
model = make_model().fit(X.iloc[tr], y.iloc[tr])

new_order = pd.DataFrame([{
    "Customer Zipcode": 725, "order_year": 2017, "order_month": 6,
    "order_week_day": 2, "order_hour": 14,
    "Shipping Mode": "Standard Class", "Market": "Europe",
    "Order Region": "Western Europe", "Category Name": "Cleats",
    "Department Name": "Recreation",   # <- does not exist in DataCo!
    "Customer Segment": "Consumer", "Type": "DEBIT",
}])
print("Late delivery probability:", round(model.predict_proba(new_order)[0, 1], 3))

Late delivery probability: 0.487


## Phase 0.6 – Rolling-origin validation and significance tests for late delivery

The models are evaluated on 6 consecutive test months (2017-04 – 2017-09). For each month, they are trained on all earlier orders and predict that month only. Months after September 2017 are excluded because the data structure changes (one item per order, sharply lower volume).

Logistic regression is the reference model. A simple rule is also included: the predicted probability is the historical late rate of the order's shipping mode. 95% confidence intervals are computed with a cluster bootstrap that resamples whole orders (1,000 replications), because items of the same order are not independent. The ROC-AUC difference to logistic regression is the main comparison; McNemar's test on the 0.5-threshold predictions is reported as a secondary test, with Holm correction.

In [6]:
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score
from xgboost import XGBClassifier
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

from src.features import delay_linear_preprocessor
from src.evaluation import rolling_origin_months, cluster_bootstrap

data_cv = data.copy()
data_cv["period"] = data_cv["order_date"].dt.to_period("M")
data_cv = data_cv[data_cv["period"] <= pd.Period("2017-09", "M")].reset_index(drop=True)

def make_models():
    return {
        "Logistic regression": Pipeline([
            ("prep", delay_linear_preprocessor()),
            ("model", LogisticRegression(max_iter=2000))]),
        "Random Forest": Pipeline([
            ("prep", delay_preprocessor()),
            ("model", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))]),
        "XGBoost": Pipeline([
            ("prep", delay_preprocessor()),
            ("model", XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                                    random_state=42, n_jobs=-1, eval_metric="logloss"))]),
    }

folds = []
for month, tr, te in rolling_origin_months(data_cv, "2017-04", "2017-09"):
    train_m, test_m = data_cv.loc[tr], data_cv.loc[te]
    out = test_m[["Order Id", "period", DELAY_TARGET]].copy()

    # Simple rule: historical late rate of the shipping mode (from training data only)
    rate = train_m.groupby("Shipping Mode")[DELAY_TARGET].mean()
    out["Shipping mode rule"] = (test_m["Shipping Mode"].map(rate)
                                 .fillna(train_m[DELAY_TARGET].mean()).values)

    for name, model in make_models().items():
        model.fit(train_m[DELAY_FEATURES], train_m[DELAY_TARGET])
        out[name] = model.predict_proba(test_m[DELAY_FEATURES])[:, 1]
    folds.append(out)
    print(month, "done")

cv = pd.concat(folds, ignore_index=True)
print("Folds:", cv["period"].nunique(), "| Test rows:", len(cv), "| Orders:", cv["Order Id"].nunique())

2017-04 done
2017-05 done
2017-06 done
2017-07 done
2017-08 done
2017-09 done
Folds: 6 | Test rows: 31292 | Orders: 10445


In [7]:
MODELS = ["Shipping mode rule", "Logistic regression", "Random Forest", "XGBoost"]
y_cv = cv[DELAY_TARGET]

def acc(y_true, p):
    return accuracy_score(y_true, p >= 0.5)

def rec(y_true, p):
    return recall_score(y_true, p >= 0.5)

rows = []
for m in MODELS:
    a, a_lo, a_hi = cluster_bootstrap(y_cv, cv[m], cv["Order Id"], acc, n_boot=1000)
    auc, auc_lo, auc_hi = cluster_bootstrap(y_cv, cv[m], cv["Order Id"], roc_auc_score, n_boot=1000)
    rows.append({"model": m,
                 "accuracy": a, "acc_CI": f"[{a_lo:.3f}, {a_hi:.3f}]",
                 "recall_late": rec(y_cv, cv[m]),
                 "roc_auc": auc, "auc_CI": f"[{auc_lo:.3f}, {auc_hi:.3f}]"})

pd.DataFrame(rows).round(3)

,model,accuracy,acc_CI,recall_late,roc_auc,auc_CI
0,Shipping mode rule,0.692,"[0.683, 0.702]",0.538,0.724,"[0.713, 0.735]"
1,Logistic regression,0.703,"[0.694, 0.713]",0.559,0.740,"[0.731, 0.750]"
2,Random Forest,0.687,"[0.679, 0.695]",0.661,0.756,"[0.747, 0.765]"
3,XGBoost,0.707,"[0.698, 0.717]",0.613,0.763,"[0.753, 0.772]"


In [8]:
REF = "Logistic regression"
rows = []
for m in ["Shipping mode rule", "Random Forest", "XGBoost"]:
    d, lo, hi = cluster_bootstrap(y_cv, cv[m], cv["Order Id"], roc_auc_score,
                                  pred_b=cv[REF], n_boot=1000)
    # McNemar: do the two models make errors on different orders?
    ok_m = (cv[m] >= 0.5) == y_cv
    ok_ref = (cv[REF] >= 0.5) == y_cv
    table = [[np.sum(ok_m & ok_ref), np.sum(ok_m & ~ok_ref)],
             [np.sum(~ok_m & ok_ref), np.sum(~ok_m & ~ok_ref)]]
    p = mcnemar(table, exact=False, correction=True).pvalue
    rows.append({"model": m,
                 "acc_diff_vs_LR": acc(y_cv, cv[m]) - acc(y_cv, cv[REF]),
                 "AUC_diff_vs_LR": d, "CI_low": lo, "CI_high": hi,
                 "p_mcnemar": p})

comparison = pd.DataFrame(rows)
comparison["p_holm"] = multipletests(comparison["p_mcnemar"], method="holm")[1]
comparison["significant"] = comparison["p_holm"] < 0.05
comparison.round(4)

,model,acc_diff_vs_LR,AUC_diff_vs_LR,CI_low,CI_high,p_mcnemar,p_holm,significant
0,Shipping mode rule,-0.0109,-0.0158,-0.0239,-0.0077,0.0000,0.0000,True
1,Random Forest,-0.0166,0.0156,0.0077,0.0230,0.0000,0.0000,True
2,XGBoost,0.0041,0.0225,0.0151,0.0298,0.0034,0.0034,True
